In [1]:
#setting libraries
import os
import re
import datetime

Data overview

In [ ]:
#reading data and separating data into test and bench files
tests_files=[a for a in os.listdir('./data.nogit/') if 'test' in a]
bench_files=[a for a in os.listdir('./data.nogit/') if 'bench' in a]
# os.getcwd()
#overview of the test files
bits={} #key:file name value:number of bits

for a in tests_files:
    total_bits=0
    n_input=0
    n_outpus=0
    with open(f'./data.nogit/{a}','r') as file:
        while True:
            line=file.readline().strip()
            if not(line):
                break
            else: #there's available data to read
                total_bits=max(total_bits,len(line))
                n_outpus=line.count('-')
                n_input=len(line)-n_outpus
    bits[a]={"total_bits":total_bits,"INPUT":n_input,"OUTPUT":n_outpus}
    file.close()

for a,circ_struc in bits.items():
    print(f'file: {a}, total bits to work: {circ_struc}')
def extract_number_from_string(string:str):
    '''
    :param str string: the string of values to evaluate
    input string to extract the integer value
    '''
    print(string)
    aux=re.findall(r'(\d+) \w+',string)
    print(aux)
    return int(aux[0])
#overview of bench files

n_vales={} #key:file_name value:{key: input/output/gates value:read value from file}
exp_values={} #key: input/output/gates value:expected number
for a in bench_files:
    n_vales[a]={}
    exp_values[a]={}
    aux=0
    print(f'read file: {a}')
    with open(f'./data.nogit/{a}','r') as file:
        while True:
            line=file.readline().strip()
            if aux>3:
                break
            if not(line):
                aux+=1           
                continue
            else:
                if line[0]=='#': #it show the information about the file
                    try:
                        exp_values[a][line.split(' ')[2].upper()]=extract_number_from_string(line.lower())
                    except:
                        continue
                else:
                    gate_str="".join(filter(str.isalpha,line))
                    if gate_str not in n_vales[a]:
                        n_vales[a][gate_str]=0
                    n_vales[a][gate_str]+=1
    file.close()

gates=[] #getting the names of the gates used in the bench files
for a,circ_struc in n_vales.items():
    print (a,circ_struc)
    for hierarchy  in circ_struc.keys():
        if hierarchy not in ['INPUT','OUTPUT']:
            if hierarchy not in gates:
                gates.append(hierarchy)

print(gates)

Code for running tests

In [4]:
#setting time to Japan time (UTC+9)
jpTime=datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9)))
#when testing started
t0=jpTime.now()
def log(msg:str):
    '''
    :param str msg: the message to appear in the log
    '''
    delta=jpTime.now()-t0 #show the running time
    print(f'{delta}...{msg}')
def logic_gate(gate:str, inputs:list): 

    '''     defining the logic gate functioning 

    Params: 
        gate(str): 'NAND', 'AND', 'OR', 'NOT', 'NOR', 'BUFF', 'XOR', 'XNOR' name of gates in bench files 
        inputs(list): defines the inputs for the gate [A,B,C...], the number of elements in the function will determine the amount of pins for the logic gate.EXCEPT for NOT gates it will take ONLY the first element 

    Returns: 
        output(bool): a single logic value to assign to the corresponding  

    ''' 

    #verifying that the input list is just integers 0/1 

    aux,inputs=inputs,[] 
    inputs=[int(x) for x in aux] 
    holder=inputs[0] 
    if gate.upper()=='NAND': 
        [holder:=holder&x for x in inputs] 
        holder= ~holder
    elif gate.upper()=="AND":
        [holder:=holder&x for x in inputs] 
    elif gate.upper()=="OR": 
        holder =0 
        [holder:=holder|x for x in inputs] 
    elif gate.upper()=="NOT":
        holder = ~holder
    elif gate.upper()=="NOR": 
        holder =0 
        [holder:=holder|x for x in inputs] 
        holder=~holder
    elif gate.upper()=="XOR": 
        holder=0 
        [holder:=holder^x for x in inputs] 
    elif gate.upper()=="XNOR": 
        holder=0 
        [holder:=holder^x for x in inputs] 
        holder=~holder

    return holder 

In [5]:
#circuit code
def circuit_code(signal_line:dict, circ_struc:list, stuck_at:dict={}):
    """This runs the circuit in fault-free mode and stuck-at fault

    Args:
        signal_line (dict): Imports what the signal logic value must be
        circ_struc (list): Which logic gate and the relationship between inputs and outputs
        stuck_at (dict): If provided, these values are flipped to whichever value is in this dict #key=signal index value=logic value to be stuck at
    Returns:
        signal_line (dict): returns the updated signal logic value for any situation (fault-free or stuck-at fault)
    """   
    for a in circ_struc:
        output_signal=a.split(' = ')[0]
        gate=a.split(' = ')[1].split('(')[0]
        input_signal_index=a.split(' = ')[1].split('(')[1][:-1]

        input_signals=[]
        for x in input_signal_index.split(', '):
            if x in stuck_at:
                 input_signals.append(stuck_at[x.strip()])
            else:
                input_signals.append(signal_line[x.strip()])
        signal_line[output_signal]=logic_gate(gate,input_signals)
        if stuck_at and (output_signal in stuck_at.keys()):
                signal_line[output_signal]=stuck_at[output_signal]
    return signal_line

In [6]:
def ex_truth_table(primary_inputs:list, primary_outputs:list,circ_struc:list):
    '''
    Params:
        primary_inputs(list): A list of the primary input signals index
        primary_outputs(list): A list of the primary output signals index
        circ_struct(list): A list with the relationships between inputs and outputs
    Returns:
        fault_free_circuit(dict): key:binary input for primary inputs value:fault-free output
    '''
    log("Beginning truth table generation")
    t0=datetime.datetime.now()
    b={} #key: signal #value:
    fault_free_circuit=[]
    for a in range(2**len(primary_inputs)):
        aux=0
        c=bin(a)[2:].zfill(len(primary_inputs)) #transform to binary 
        while aux<len(primary_inputs):
            b[primary_inputs[aux]]=int(c[aux])
            aux+=1
        print(b)
        normal_output=circuit_code(b,circ_struc)
        bench_outputs=[normal_output[x] for x in primary_outputs]
        # print(bin(a)[2:].zfill(3),bench_outputs)
        fault_free_circuit.append(bench_outputs)
    log("Finished generating truth table")
    print(f'Time take bit logic vector: {datetime.datetime.now()-t0}')
    return fault_free_circuit

In [7]:
#reading the bench file to assign the input, output and intermediate connections.
#read only once (the rutine is the same for ALL the data)
def read_bench(bench_file:str):
    """_summary_

    Args
    -------
        bench_file : str
            the path of the bench file to evaluate

    Returns
    -------
        circ_struct: list
            relationship input, gate and output

        signal_hierarchy: dict
            signals map of the circuit
        
        truth table: dict
            ONLY if the circuit has 8 or less inputs, the truth table is generated
    """      
    aux=0
    circ_struc=[] #circuit structure
    signal_hierarchy={} #key: level #value[list]: list of inputs. All signals are considered input
    signal_hierarchy[0]=[] #[0]=PI [-1]=PO
    signal_hierarchy[-1]=[]
    fanout_stems={}
    gate_inputs=0
    n_inv=0
    with open(bench_file) as f:
        while True:
            a=f.readline().strip()
            if aux>3:
                break
            if not(a):
                aux+=1
                continue
            else:
                if "#" in a:
                    continue
                else:
                    if 'INPUT' in a or 'OUTPUT' in a: #connections for primary input and primary output
                        b=a.split('(')[1][:-1]
                        if 'INPUT' in a:
                            signal_hierarchy[0].append(b)
                            fanout_stems[b]=0
                        else:
                            signal_hierarchy[-1].append(b)
                    else: # signal = gate (input signal)
                        circ_struc.append(a)
                        
                        if 'NOT' in a: #getting the number of inverters
                            n_inv+=1

                        y='0'
                        for z in a.split(' = ')[1].split('(')[1][:-1].split(', '):
                            try: #try to convert to int
                                y=max(int(y),int(z))
                            except: #works with string characters
                                y=max(y,z)
                            for x in signal_hierarchy.values():
                                if z in x:
                                    fanout_stems[z]+=1
                            if 'BUFF' not in a:
                                gate_inputs+=1
                        ind=0
                        y=str(y)
                        #evaluates in string format
                        while ind<(len(signal_hierarchy)-1):                            
                            if y not in signal_hierarchy[ind]:
                                ind+=1
                            else:
                                break
                        ind+=1
                        if ind not in signal_hierarchy:
                            signal_hierarchy[ind]=[]
                        signal_hierarchy[ind].append(a.split(' = ')[0])
                        fanout_stems[a.split(' = ')[0]]=0
    f.close()
    log (f"Finished reading bench file: {bench_file}")
    stems=0
    for y,z in fanout_stems.items():
        if z>1:
            stems+=1
    aux=0
    for z in signal_hierarchy.values():
        aux+=len(z)
    log (f'There area {aux*2} possible faulty circuits under the single-fault assumption')
    print(f'Number of PIs: {len(signal_hierarchy[0])}')
    print(f'Number of POs: {len(signal_hierarchy[-1])}')
    print(f'Number of fanout stems: {stems}')
    print(f'Number of gate inputs {gate_inputs}')
    print(f'Number of inverters: {n_inv}')
    print(f'Number of collapsed faults: {2*(len(signal_hierarchy[-1])+stems)+gate_inputs-n_inv}')


    if len(signal_hierarchy[0])<8: #it was chosen 8, in order to use an 8 switch input from wokwi
        log(f'circuit is small enough to perform a functional testing, generating truth table')
        z=truth_table(signal_hierarchy[0],signal_hierarchy[-1],circ_struc)
        return circ_struc, signal_hierarchy,z
    else:
        log(f'circuit has over 10  inputs, too large to perfom a functional testing')
        return circ_struc, signal_hierarchy 

In [ ]:
def truth_table(primary_inputs:list, primary_outputs:list,circ_struc:list):
    '''
    Params:
        primary_inputs(list): A list of the primary input signals index
        primary_outputs(list): A list of the primary output signals index
        circ_struct(list): A list with the relationships between inputs and outputs
    Returns:
        fault_free_circuit(dict): key:binary input for primary inputs value:fault-free output
    '''
    log("Beginning truth table generation")
    t0=datetime.datetime.now()
    b={} #key: signal #value:
    for a in primary_inputs:
        b[a]=[]
    fault_free_circuit=[]
    print("finished generating signal spaces")

    ## pending improvement (find formula that allow transformation directly into vector)
    for a in range(2**len(primary_inputs)):
        aux=0
        c=bin(a)[2:].zfill(len(primary_inputs)) #transform to binary 
        while aux<len(primary_inputs):
            # b[primary_inputs[aux]].insert(0,c[aux])
            b[primary_inputs[aux]].append(c[aux])
            aux+=1
    print(b)
    
    #####
    for a in b.keys():
        b[a]=int(''.join(b[a]),2)
    print(b)
    normal_output=circuit_code(b,circ_struc)
    bench_outputs=[normal_output[x] for x in primary_outputs]
    print(bench_outputs)#,bin(bench_outputs)[2:].zfill(len(primary_inputs)))
    # fault_free_circuit.append(bench_outputs)
    log("Finished generating truth table")
    print(f'Time take bit logic vector: {datetime.datetime.now()-t0}')
    return fault_free_circuit

In [11]:
a,b,c=read_bench(
    './data.nogit/c1.bench'
)


0:01:08.676323...Finished reading bench file: ./data.nogit/c1.bench
0:01:08.676403...There area 20 possible faulty circuits under the single-fault assumption
Number of PIs: 3
Number of POs: 1
Number of fanout stems: 1
Number of gate inputs 7
Number of inverters: 1
Number of collapsed faults: 10
0:01:08.676483...circuit is small enough to perform a functional testing, generating truth table
0:01:08.676498...Beginning truth table generation
finished generating signal spaces
{'a': ['0', '0', '0', '0', '1', '1', '1', '1'], 'b': ['0', '0', '1', '1', '0', '0', '1', '1'], 'c': ['0', '1', '0', '1', '0', '1', '0', '1']}
{'a': 15, 'b': 51, 'c': 85}
[71]
0:01:08.676599...Finished generating truth table
Time take bit logic vector: 0:00:00.000102


In [12]:
ex_truth_table(b[0],b[-1],a)

0:01:10.153107...Beginning truth table generation
{'a': 0, 'b': 0, 'c': 0}
{'a': 0, 'b': 0, 'c': 1, 'd': 0, 'e': 0, 'f': -1, 'h': 0, 'g': 0, 'y': 0}
{'a': 0, 'b': 1, 'c': 0, 'd': 0, 'e': 0, 'f': -1, 'h': 1, 'g': 0, 'y': 1}
{'a': 0, 'b': 1, 'c': 1, 'd': 1, 'e': 1, 'f': -2, 'h': 0, 'g': 0, 'y': 0}
{'a': 1, 'b': 0, 'c': 0, 'd': 1, 'e': 1, 'f': -2, 'h': 0, 'g': 0, 'y': 0}
{'a': 1, 'b': 0, 'c': 1, 'd': 0, 'e': 0, 'f': -1, 'h': 0, 'g': 0, 'y': 0}
{'a': 1, 'b': 1, 'c': 0, 'd': 0, 'e': 0, 'f': -1, 'h': 1, 'g': 0, 'y': 1}
{'a': 1, 'b': 1, 'c': 1, 'd': 1, 'e': 1, 'f': -2, 'h': 0, 'g': 1, 'y': 1}
0:01:10.153708...Finished generating truth table
Time take bit logic vector: 0:00:00.000491


[[0], [1], [0], [0], [0], [1], [1], [1]]

In [13]:
truth_table(b[0],b[-1],a)

0:01:11.392496...Beginning truth table generation
finished generating signal spaces
{'a': ['0', '0', '0', '0', '1', '1', '1', '1'], 'b': ['0', '0', '1', '1', '0', '0', '1', '1'], 'c': ['0', '1', '0', '1', '0', '1', '0', '1']}
{'a': 15, 'b': 51, 'c': 85}
[71]
0:01:11.392657...Finished generating truth table
Time take bit logic vector: 0:00:00.000115


[]

In [ ]:
#reading the test file for data and testing
#this script produces the "correct" behaviour
def read_test(test_file: str,circ_struc: list,signals:dict):
    """reads the test vector form the test file and generates the corresponding output

    Args:
        test_file (str): file with test vectors
        circ_struc (list): It contains the information of the logic gates and it inputs and outputs signals
        signals (dict): It has the hierarchy of all signal where [0]= primary inputs and [len(signals)-1]= primary outputsca

    Returns:
        _type_: _description_
    """    
    primary_input=signals[0]
    primary_output=signals[len(signals)-1]
    signal_line={}
    with open(test_file) as f:
        while True:
            line=f.readline().strip()
            if not(line):
                break
            else: #I assume the data is organized in the bench stated order
                #assigning the test data to the corresponding signals
                for a in range(len(primary_input)):
                    try:
                        signal_line[primary_input[a]]=int(line[a])
                    except:
                        continue
                
                #executing the testing from the bench file (running the circuit)
                #structure= output signal= logic gate (input signal)
                signal_line=circuit_code(signal_line,circ_struc)
                
                circuit_output=[signal_line[x] for x in primary_output]
                print(f'Data input: {line[:line.index('-')]}, \tData output: {circuit_output}')
    f.close()
    log(f"Finished running test file {test_file}")
    return circuit_output

Code to draw the circuit in the Wokwi

In [ ]:

#logic gates codes to draw circuits in wokwi
def logic_gate_diagram(gate:str,number:int,top:int,left:int):
    '''
    Params
    -------
        gate(str): the logic gate desired
        number(int): the number for id
        top(int): top value for wokwi reference system
        left(int): left value for wokwi reference system
    Returns
    -------
        part_json(str): the corresponding json to draw the circuit on Wokwi
        output_json(str): the correspondin json to indicate the ouput\
    '''
    if gate=='NAND':
        type="wokwi-gate-nand-2"
    elif gate=='AND':
        type="wokwi-gate-and-2"
    elif gate=='NOR':
        type= "wokwi-gate-nor-2"
    elif gate=='OR':
        type="wokwi-gate-or-2"
    elif gate=='XOR':
        type="wokwi-gate-xor-2"
    elif gate=='XNOR':
        type="wokwi-gate-xnor-2"
    elif gate=='NOT':
        type="wokwi-gate-not"
    elif gate=='BUFF':
        type="wokwi-gate-buffer"
        
    return f'"type":"{type}", "id":"{gate.lower()}{str(number)}","top":{top},"left":{left}'

def signal_text(id:int,top:int ,left: int):
    return  f'"type": "wokwi-text", "id": "text{id}", "top": {top}, "left": {left+100}, '

def gate_left_top(circ_struc:list,hierarchy:dict):
    """Determines the top and left coordinates of the logic gates

    Args:
        circ_struc (list): _description_
        hierarchy (dict): _description_

    Returns:
        rel_pos (dict): A dictionary with the following structure <br>Key:signal <br>Value (list):[left coordinate, top coordinate]
    """
    rel_pos={} #key: signal #value[left,top]
    aux=0
    #set left position
    for y,z in hierarchy.items():
        for x in z:
            rel_pos[x]=[y]
            if y==0:
                rel_pos[x].append(aux)
                aux+=1

    for y in circ_struc:
        t=y.split(' = ')[0]
        u=y.split(' = ')[1].split('(')[1][:-1].split(', ')
        w=0
        for x in u:
            w+=rel_pos[x][1]
        rel_pos[t].append(w/len(u))
    return rel_pos

In [ ]:
#logic gates connections
#[start, finish, color, ["h","v"]]
[ "xor0:OUT", "xor4:A", "blue", [ "v0" ] ],
[ "xor2:OUT", "xor4:B", "red", [ "v0" ] ],
[ "xor5:A", "xor2:OUT", "red", [ "h0" ] ],
[ "xor3:OUT", "xor5:B", "#fc23fc", [ "v0" ] ],
[ "xor2:B", "xor1:OUT", "#4e23fc", [ "h0" ] ],
[ "xor3:A", "xor1:OUT", "#4e23fc", [ "h0" ] ],
[ "sw1:1b", "xor0:A", "white", [ "v0" ] ],
[ "xor2:A", "sw1:2b", "green", [ "h0" ] ],
[ "sw1:3b", "xor0:B", "#a84032", [ "h0" ] ],
[ "xor1:A", "sw1:3b", "#a84032", [ "h0", "v0" ] ],
[ "sw1:6b", "xor1:B", "#fcfc23", [ "h30", "v60" ] ],
[ "sw1:7b", "xor3:B", "#23fcc9", [ "v0" ] ]

In [ ]:
a,b,c=read_bench('./kelvin_testing/c17_XNOR.bench')
rel_pos=gate_left_top(a,b)

for d,e in enumerate(a):
    g_out=e.split(' = ')[0]
    print('{',signal_text(d,rel_pos[g_out][1]*100,rel_pos[g_out][0]*120),'"attrs": {"text":"',g_out,'"} },')

In [ ]:
for d,e in enumerate(a):
    #determine logic gate
    gate=e.split(' = ')[1].split('(')[0]
    
    #determine the output
    g_out=e.split(' = ')[0]
    print("{",logic_gate_diagram(gate,d,rel_pos[g_out][1]*(100),rel_pos[g_out][0]*120),"},")
print('{ "type": "wokwi-dip-switch-8", "id": "sw1", "top": 0, "left": 0, "rotate": 90, "attrs": {} }')#adding a switch for the ipnuts


In [ ]:
wokwi_diagram_json={
    "version":1,
    "author":"Anonymous maker",
    "editor":"wokwi",
    "parts":[], #logic gates and text
    "connnections":[],#wires between logic gates
    "dependencies":{}
}

Stuck-at fault code
Fault equivalency simulations for every logic gate

For any gate try SA0 and SA1 at both input and output

In [ ]:
#for each logic gate testing fault colapsing
def brute_SA_faults_logic_gates_colapsing(gate:str,inputs:str|int):
    '''
    Params:
        gate(str): the logic gate to evaluate
        inputs(str|int): is the number of inputs for the gate str:'0000' int:2
    Returns:
    '''

    # inputs=2
    # gate='NAND'
    if isinstance(inputs,int):
        lenght=inputs
    elif isinstance(inputs,str):
        lenght=len(inputs)
    else:
        # return "the provided input is not in a valid format"
        print("the provided input is not in a valid format")

    test='SA0'
    failure=''
    signal=['A','B','Output']
    #SA for the inputs
    for a in range(2**lenght+2):
        for b in range(2**lenght):
            pi_input=bin(b)[2:].zfill(lenght)
            po=logic_gate(gate,list(pi_input))
            sa_input=pi_input[0:(a//2)]+str(int(test[-1]))+pi_input[(a//2)+1:]
            if a<2**lenght:
                sa_output=logic_gate(gate,list(sa_input))
            else:
                sa_input='xx'
                sa_output=int(test[-1])
            
            if sa_output!=po:
                failure="failed"
            else:
                failure=''
            print(f'|{gate}|{pi_input}|{test}[{signal[a//2]}]|{sa_input}|{po}|{sa_output}|{failure}|')
        if test=='SA0':
            test='SA1'
        else:
            test='SA0'

In [ ]:
#for each logic gate testing fault colapsing
def brute_SA_faults_logic_gates_comparing(gate1:str,gate2:str,inputs:str|int):
    '''
    Params:
        gate(str): the logic gate to evaluate
        inputs(str|int): is the number of inputs for the gate str:'0000' int:2
    Returns:
    '''

    # inputs=2
    # gate='NAND'
    if isinstance(inputs,int):
        lenght=inputs
    elif isinstance(inputs,str):
        lenght=len(inputs)
    else:
        # return "the provided input is not in a valid format"
        print("the provided input is not in a valid format")

    test='SA0'
    failure=''
    signal=['A','B','Output']
    #SA for the inputs
    for a in range(2**lenght+2):
        for b in range(2**lenght):
            pi_input=bin(b)[2:].zfill(lenght)
            po1=logic_gate(gate1,list(pi_input))
            po2=logic_gate(gate2,list(pi_input))
            sa_input=pi_input[0:(a//2)]+str(int(test[-1]))+pi_input[(a//2)+1:]
            if a<2**lenght:
                sa_output1=logic_gate(gate1,list(sa_input))
                sa_output2=logic_gate(gate2,list(sa_input))
            else:
                sa_input='xx'
                sa_output=int(test[-1])
            
            print(f'|{pi_input}|{test}[{signal[a//2]}]|{sa_input}|{sa_output1}|{sa_output2}|')
        if test=='SA0':
            test='SA1'
        else:
            test='SA0'

In [ ]:

#for perfoming brute SA fault 
def functional_SA_single_faults(circuit:list, hierarchy:dict):
    """performs an Stuck-at 0 and Stuck-at 1 single fault for every circuit segment
    Compares the obtained result against the truth table

    Args:
        circuit (list): relationship between logic gate, input and output
        hierarchy (dict): signal map of the circuit

    Returns:
        SA_signal_line (dict): key:signal_SA0/SA1 value:[list of output]
    """    
    log("Generating SA fault truth table")
    lenght=[]
    test_vector={}
    SA_signal_line={}
    for z in hierarchy.values():
        for y in z:
            lenght.append(y)
    #SA for the inputs
    for a in lenght:
        for test in ['SA0','SA1']:
            SA_signal_line[f'{a}_{test}']=[]
            for b in range(2**len(hierarchy[0])):
                aux=0
                c=bin(b)[2:].zfill(len(hierarchy[0]))
                while aux<len(hierarchy[0]):
                    test_vector[hierarchy[0][aux]]=int(c[aux])
                    aux+=1
                # print(test_vector,circuit,{a:int(test[-1])})
                outputs=circuit_code(test_vector,circuit,{a:int(test[-1])})
                SA_signal_line[f'{a}_{test}'].append([outputs[x] for x in hierarchy[len(hierarchy)-1]])
    log("Finished generating SA fault truth table")
    return SA_signal_line

def comparte_truth_SA_tables(fault_free_truth_table:list, SA_truth_table:dict, hierarchy:dict, show:str ='y',file:str='null'):
    """compares the results from the fault free and SA fault truth table

    Args:
        fault_free_truth_table (list): fault free truth table results. It's assumed that the index of the list corresponds to the binary input
        SA_truth_table (dict): SA fault truth table result. Key:(signal_SA0/SA1) Value (list):result of SA fault truth table
        hierarchy (dict): Circuit signal hierarchy
        show (str): to print result on screen
        file (str): name of the file to generate readme file in github

    Returns:
        SA_vector (dict): Table with the SA faults and test vector that generate failures at the primary outputs.
        Key: SA fault
        Value: test vector
        
        test_vector (dict):Table with the test vector and SA faults that generate failures at the primary outputs
        Key: test vector
        Value: SA fault
        
    """    
    log("Comparing fault free and functional SA fault truth tables")

    SA_vector={}
    for d in SA_truth_table.keys():
        aux=0
        SA_vector[d]=[]
        while aux<len(fault_free_truth_table):
            if fault_free_truth_table[aux]!=SA_truth_table[d][aux]:
                SA_vector[d].append(bin(aux)[2:].zfill(len(hierarchy[0])))
            aux+=1
    
    #which test vector caused SA faults
    test_vector={}
    w=0 # test vector columns
    v=0 # SA faults columns
    for y,z in SA_vector.items():
        w=max(w,len(z))
        for x in z:
            if x not in test_vector:
                test_vector[x]=[]
            test_vector[x].append(y)
            v=max(v,len(test_vector[x]))
    
    if file!='null':
        #format data table
        with open(f'{file}.md','w') as f:
            f.write('| SA fault | ')
            for u in range(w):
                f.write('test vector | ')
            f.write('\n')
            f.write ('| :---: | ')
            for u in range(w):
                f.write(' :---: | ')
            f.write('\n')
            while w>0:
                for y,z in SA_vector.items():
                    if len(z)==w:
                        f.write ('|')
                        f.write(f'{y} |')
                        for x in z:
                            f.write(f'{x}|')
                        f.write('\n')
                w-=1
            f.write('\n\n\n')

            f.write('| Test vector | ')
            for u in range(v):
                f.write('SA fault | ')
            f.write('\n')
            f.write ('| :---: | ')
            for u in range(v):
                f.write(':---: | ')
            f.write('\n')
            
            while v>0:
                for y,z in test_vector.items():
                    if len(z)==v:
                        f.write ('|')
                        f.write(f'{y}|')
                        for x in z:
                            f.write(f'{x}|')
                        f.write('\n')
                v-=1
        f.close()
        
    log("Finished comparing fault free and functional SA fault truth tables")
    return SA_vector, test_vector